In [ ]:
# hide
import numpy as np
import pyquist as pq
from pyquist.helper import pitch_name_to_pitch, pitch_to_frequency

In [ ]:
def karplus_strong(pitch: str, duration: float, f_s: int = 44100) -> pq.Audio:
    """A plucked string: a burst of white noise through a damped, low-passed feedback delay line."""
    M = round(f_s / pitch_to_frequency(pitch_name_to_pitch(pitch)))
    g = 10 ** (-3 * M / (f_s * duration))      # decay by about 60 dB over the duration
    N = int(duration * f_s)
    x = np.zeros(N)
    x[:M] = np.random.uniform(-1.0, 1.0, M)    # excitation: one period of white noise
    buffer = np.zeros(M)                       # the delay line, as a circular buffer
    i = 0
    previous = 0.0                             # the low-pass filter's memory, y[n - M - 1]
    y = np.zeros(N)
    for n in range(N):
        delayed = buffer[i]                                  # y[n - M]
        y[n] = x[n] + g * 0.5 * (delayed + previous)         # low-pass, attenuate, and add
        previous = delayed
        buffer[i] = y[n]
        i = (i + 1) % M
    return pq.Audio(y, f_s)


# Strum all six strings of a guitar (an E minor chord), one string every 40 ms
strum = pq.Score([(0.04 * i, {"pitch": pitch, "duration": 3.0})
                  for i, pitch in enumerate(["E2", "B2", "E3", "G3", "B3", "E4"])])
pq.play(strum.render(karplus_strong), normalize=True)